#Definition af CNNTRansformer klasse 

- num_classes : Number of classes we want to classify prediction in. In our case it is binary (True / False)

- in_channels : Number of columns in dataset, or number of culumns the model should analyze. In our case it is all columns, expect for timestep. According to chat, we don't need timestep to make preidiction. 

- embed_dim : size of feature vector used by transformer. if training is unstable, reduce to 64, if model overfits, use 256. Embed_dim must be divisable by num_heads (FInd ud af specifikt h)

- num_heads : Number of attention heads in transformers mult-head attention layer. 

- num_layers : Number of transformer layers (?)

- mlp_dim : Size of feedforward inside each transformor layer (Hvorfor er det 256)

- dropout : Percentage of neurons we randomly turn off during each training step. 

#Definition af CNN feature extractor

- self.cnn = nn.Sequential()
    - order of inputs define which order each layer should execute in

- nn.Conv1d(in_channels, 32, kernel_size=3, stride=2, padding=1)
    - in_channels : Number of input features, so how many variables are in each timestep
    - 32 : Output channels. How many patterns the CNN will learn.
    - Kernel_size : Size of the filter
    - Stride : how many steps we move the filter
    - padding : how many 0 we add to the start and end of the input. Prevents the sequence from shrinking too much. 

- nn.BatchNorm1d(32)
    - Normalizes the activations 
    
- nn.ReLU(inplace=True)
    - Means we are using relu
    - Inplace : modifys the current tensor directly, instead of creating a whole new tensor. This saves memory. 

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

path = Path('datasets/EPIC/Scenario_1/EpicLog_Scenario 1_19_Oct_2018_14_44.csv')

# Load the CSV file
df = pd.read_csv(path)
df = df.drop(columns=['Timestamp'])
df

#print(df.dtypes[df.dtypes == "object"])

In [ ]:
#Test vidreudviklng
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNTransformer(nn.Module):
    def __init__(
        self,
        in_channels: int,
        embed_dim: int = 128,
        num_heads: int = 4,
        num_layers: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.encoder = nn.Sequential(

            #Første convolutional lag
            #den ser på smp vinduer i tiddserien og lærer lokale/korte mønstre
            #Så vi går fra raw data af signaler og laver det til et lag af features/mønstre
            nn.Conv1d(in_channels, 32, kernel_size=5, padding=2),

            #Relu tilføjer non-linearity
            #Dette gør at modellen er i stand til at lærer komplekse mønstrer og ikke bruger en linær funktion til at forstå vores features 
            nn.ReLU(),

            #Halverer sekvens længden og sletter unødvendig støj(?) så sekvensen er nemmere at analuserer senere 
            #SÅ hvis længden er 256 før, bliver den til 128 efter
            nn.MaxPool1d(2),


            #Anden convlutional blok
            #Modellen bygger videre på de første features og lærer mere advencerede og langvarige mønstrer
            #efter anden pooling, bliver sekvens længden endy kortere, så f. eks. 128 bliver til 64
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),


            # Sidste convolutional blok, hvor vi faktisk danner et output. Derfor har vi ikke maxpool i dette lag
            #Her mapper vi vores features til en embedded dimmension. I vores tilfælde er den dimmension 128
            #Dette er den sequence vores transformer senere skal arbejde på. 
            nn.Conv1d(64, embed_dim, kernel_size=5, padding=2),
            nn.ReLU(),

            #Outputtet af encoderen vil være : [batch, embed_dim, reduced_seq_len]
            # dette skal representerer vores fuldendte encoded features 
        )

        # Transformer, der arbejder på z
        
        #Her definerer vi bare hvordan vores TRANSFORMER encoder lag ser ud 
        #nn.transformerEncoderLayer gør 
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dropout=dropout,
            batch_first=True #Definerer at input skal være af formen [batch, seq_len, embed_dim]
        )

        #Her laver vi en stack af flere transformer encoder lag. 
        # så hvis vi definerer at num layers er 2, så går vores input sekvens genne 2 encoder lag
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers

            #Resultatet af denne funktion er z' hvilket er en sekvens med transformer encoded features
            #Så det representerer vores encoded features med større context mønstrer
        )

        
        self.decoder = nn.Sequential(
            #Den første decoder convulitonal blok oversætter features tilbae mod signal rummet? 
            nn.Conv1d(embed_dim, 64, kernel_size=5, padding=2),
            nn.ReLU(),

            #Andet decoder lag fortætter med rekonstruktionen, for at deres det mere præcist 
            nn.Conv1d(64, 32, kernel_size=5, padding=2),
            nn.ReLU(),

            #Sidste decoder lag 
            # dette lag mapper vores features tilbage til samme antal kanaler som vores oprindlige input havde
            nn.Conv1d(32, in_channels, kernel_size=5, padding=2),

            #bemærk at sekvenslængden stadig er kortere end det oprindlige input tho. 
            #outputtet af decoderen er så x_hat, hvilket er det reconstructet signal. 
        )


    # x er vores input til modellen, og representerer ...? 
    # x ville typisk have denne shape : [batch, in_channels, seq_len]
    # så batch = hvor mange samples vi sender ind af gangen, in_channels = mængden af features pr timestep, seq_len = længden af timeseries?
    def forward(self, x: torch.Tensor) -> torch.Tensor:

        #Starter ud med encoder 
        #encoderen Er vores feature extractor
        #Finder lokale mønstre/features med conv1d og gør seqencen kompakt med MaxPool1d
        z = self.encoder(x)              # [batch, embed_dim, reduced_seq_len]

        # Transformer forventer [batch, seq_len, embed_dim], men vi for en anden output fra encoderen
        # Så vi transposer vores encoder output så den passer ind i transformer metoden. 
        z = z.transpose(1, 2)            # [batch, reduced_seq_len, embed_dim]
        z = self.transformer(z)          # [batch, reduced_seq_len, embed_dim]

        #Så transformer vi vores output tilbage til den oprindlige form, så vi kan bruge den i vores decoder, siden decoderen er en conv1d decoder. 
        z = z.transpose(1, 2)            # [batch, embed_dim, reduced_seq_len]

        #Så har vi vores decoder, som prøver at omsætte vores transformer features tilbage til noget der ligner det oprindlige signal
        #Altså den reconstructor vores sekvens til noget der viser de mønstrer der er blevet udregnet i vores encoders 
        x_hat = self.decoder(z)          # [batch, in_channels, reduced_seq_len]

        #Dette er så upsampling / Interpolation
        #Dette gør at sekvenslængen bliver ligeså stor som den oprindlige input var 
        #Dette er nødvendigt fordi vi har lavet en "Auto encoder" som forventer at output længden er det same som inputtet
        #En autoencoder er baisically at den træner ved at minimerer forskellen mellem x og x_hat, så det er vigtigt at de er samme længde og samme form så de bliver compared ordenligt. 
        x_hat = F.interpolate(
            x_hat,
            size=x.shape[-1],
            mode="linear",
            align_corners=False
        )                                # [batch, in_channels, original_seq_len]

        return x_hat

I test eksempelet ovenover kører vi med denne rækkefølge : 

x -> CNN encoder -> z -> Hugging Face BERT -> z' -> decoder -> x_hat

Hvor : 
- z = encoded features
- z' = transformer processed features 
- x_hat = reconstructed signal 

Så : 
- Input kommer med x, som er ...
- Encoder virker som vores feature extractor 
- Transformer lærer relationer mellem time series i de extracted features 
- Decoder prøver at rekonstruerer signalet? 
- x_hat er så det rekonstruerede output 

In [ ]:
#Hvordan man plotter en feature? (Måske er det nydvendigt senere)
df_numeric = df.apply(pd.to_numeric, errors="coerce")

plt.plot(df_numeric.iloc[30:, 0])  # Plot the first feature (column index 0) for all rows except the header
plt.xlabel("timestep")
plt.ylabel("Feature value")
plt.title("Row 0 features")
plt.show()

print(df_numeric.dtypes)

#Transformer configuration 
- Config : Gets the  configuration of the pretrained model, in our case bert, which is from huggingface
- Model : generates the model from the configuration, initialized with randomized weights

The config, defines the following:
- hidden size : size of the vector used to represent each token inside the model. So the size of how many numbers should represent a token/word, and that becomes a vector. Must be divisible by number of heads. 

- number of layers : Number of transformer blocks stacked on top of eachother. Contains mult head layer connections. 12 layers. 

- number of attention heads : The amount of attention heads running in parralel. So each head would focus on different attention pattern (Long term relation, grammar, closely-related, etc.)

- intermediate size : hidden size of the feedforward network inside each tranformer network. Each layer has a small neural network. 
    - linear layer -> Activation function -> Linear layer
    - Often size = 4 * hidden size?

- dropout values : Regularization technique. Randomly turns of a percentage of random neurons. 

- vocabulary size : How many unqiue tokens a language knows. 

- positional embedding length : Defines maximum sequence length. So a model can only process at most x amount of tokens at once. Each position has a vector. For time series models this corresponds to max window size (sequence length).
    - final embedding = token embedding + positional embedding.




In [ ]:
#Calculation af loss
#loss definerer hvor forkert vores models predictions er 



#Her definerer vi vores loss function
#CrossEntropyLoss bruges til multi class klassifikation 
#(Er det relevant for os, hvis vi bare skal have normal / anomoly classification?)
#Anywayl, denne funktion forventer et input logits = [batch_size, num_classes], og labels: [batch_size]
criterion_test = nn.CrossEntropyLoss()

#Istedet bruger vi mean squared Error loss, fordi crossEntropyLoss er faktisk kun godt hvis det er supervised dataset. 
#Og det er fordi den bruger labels, som kun er med i supervised dataset
criterion = nn.MSELoss()

#Vi laver vores model, som vi definererede før
#Vores model kræver et input in_channels. 
model = CNNTransformer(in_channels=1)

#Vi konverterer alle værdier i vores dataset til float, så alle booleans bliver enten til 1 eller 0
df_numeric = df_numeric.astype(float)


#test til at loss ikke er NaN
df_numeric = df_numeric.fillna(0.0)

# Normalisering
mean = df_numeric.mean()

# Undgå division med 0
std = df_numeric.std().replace(0, 1)

df_numeric = (df_numeric - mean) / std

# safety
df_numeric = df_numeric.fillna(0.0)




#VI definerer så vores input data
#Her konverterer vi vores data til en pytorch tensor
X = torch.tensor(df_numeric.values, dtype=torch.float32)  # shape: (num_samples, 1, num_features)
#DOg siden vores cnn forventer 3 dimmensioner, og vi mangler en kanal dimmension,
#Bruger vi unsqueeze, så x går fra at være [num_samples, num_features] til at være [num_samples, 1, num_features]
X = X.unsqueeze(1)

#Dette er så vores labels 
#Dog siden EPIC er et unsupervised dataset, er der ingen labels i dataet. 
#y = torch.tensor(df_labels.values, dtype=torch.long)

#Dette er så vores forward pass
#VI sender inputtet gennem modellen 
# X → CNN → Transformer → classifier → x_hat
#Outputtert bliver [batch_size, num_classes]
#SÅ x_hat er bare rå "scores" fra modellen, men det er ikke probabilities endnu!!!!
#Probabilites sker først når vi kører softmax, hvilket sker INDE i Cross entropy loss
x_hat = model(X)    

#Vi udregner så loss nu
#Inde i crossEntropyLoss, bliver der kørt Softmax(x_hat)
#Her for vi "negative log likelyhood" og "Average loss"
#Så hvis alt er godt, skal average loss blive mindre og mindre 

#Det dog siden vi bruger MSE, så er der faktisk ingen softmax som der sker. 
#Dette er fordi der ikke er noget classification, fordi vi ikke har nogen labels. 
#Istedet så finder vi forskellen mellem modelens input out output.
#Så vi kvadrer forskellen og tager genngemsnittet af det. 
#Så baseret på denne forskel, finder vi hvad lossen er.
#Så hvis X(Inout) er tæt på x_hat(output), er der lille loss.
#Så hvis forskellen mellem X og x_har er stor så er det en anomaly. 
loss = criterion(x_hat, X)

print(loss)

In [ ]:
#træning 
#Vi SKAL have en testing metode, så den kan blive kaldt i virtual machines til vores federated learning
#Vi skal også have en evaluate function!!!!
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# preprocess data
df_numeric = df.apply(pd.to_numeric, errors="coerce")
df_numeric = df_numeric.astype(float)
df_numeric = df_numeric.fillna(0.0)

mean = df_numeric.mean()
std = df_numeric.std().replace(0, 1)
df_numeric = (df_numeric - mean) / std
df_numeric = df_numeric.fillna(0.0)

# tensor
X = torch.tensor(df_numeric.values, dtype=torch.float32)
X = X.unsqueeze(1)
X = X.to(device)

# model
model = CNNTransformer(
    in_channels=1,
    embed_dim=128,
    num_heads=4,
    num_layers=2,
    dropout=0.1
).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

epochs = 20

for epoch in range(epochs):
    model.train()

    x_hat = model(X)
    loss = criterion(x_hat, X)

    if torch.isnan(loss) or torch.isinf(loss):
        print(f"Loss exploded at epoch {epoch+1}")
        break

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6f}")